# DB7 — original CNN versus amplitude + velocity preprocessing

This is a controlled test inspired by the amplitude–velocity idea in Bian et al.,
*Interpretable Shared-Backbone MobileViT Framework* (2026), Sections 2.3–2.4.
It is **not T-EKIM**: there are no 2D histograms, grid weighting, image tiling or MobileViT.

The original baseline scored 90.19% mean validation accuracy in the previous
diagnostic run; the rejected dual-stream model scored 87.06%. This notebook
keeps the original CNN and appends 12 standardized first-difference EMG channels.
EMG and ACC are retained, with 400 ms windows and the existing 4/1/1 split.
Both models are retrained to provide a matched comparison, not compared to old test scores.

**Kaggle:** attach the same DB7 dataset, enable GPU, Save Version → Run All.
Default: 22 subjects × 2 variants = 44 fits. Use `RUN_SUBJECTS=[1]` to check execution.
The synthetic preflight tests preprocessing, training, checkpoint reload and output writers.
Download `db7_amplitude_velocity_<timestamp>.zip` from Kaggle Output.

Derivative scaling is fitted on training windows only and saved with the checkpoint.
Derivatives never cross a window boundary. All original diagnostic reports remain.
No test predictions, final refit, extra denoising or optimizer changes are introduced.
The new input weights add 3,816 trainable parameters for the standard 48-channel input.

Source: https://doi.org/10.1002/cpe.70916


In [ ]:
import os, gc, time, json, warnings, random, re, shutil
from pathlib import Path
from copy import deepcopy
from math import gcd

import numpy as np
import pandas as pd
from scipy import io
from scipy.signal import (
    resample_poly, butter, sosfiltfilt, iirnotch, filtfilt
)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve,
)

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

warnings.filterwarnings('default')
import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Configuration — matched to the leakage-safe DNN/GNN within-subject protocol


In [ ]:
def _find_kaggle_input() -> Path:
    base = Path('/kaggle/input')
    if not base.exists():
        return Path('/kaggle/input/ninapro-db7/Dataset')

    def _has_subjects(p):
        return p.is_dir() and any(
            c.is_dir() and c.name.lower().startswith('subject_')
            for c in p.iterdir()
        )

    def _search(root, depth=0):
        if depth > 5:
            return None
        if _has_subjects(root):
            return root
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    result = _search(child, depth + 1)
                    if result is not None:
                        return result
        except PermissionError:
            pass
        return None

    result = _search(base)
    return result if result else Path('/kaggle/input/ninapro-db7/Dataset')


class Config:
    KAGGLE_INPUT = _find_kaggle_input()
    KAGGLE_WORKING = (
        Path('/kaggle/working')
        if Path('/kaggle').exists()
        else Path.cwd() / 'mka_cnn_within_working'
    )

    # Keep these choices fixed before final test evaluation.
    # Matched preprocessing comparison: keep both modalities and all other settings fixed.
    DIAG_EXPERIMENTS = [('baseline', 'both'), ('amplitude_velocity', 'both')]
    DIAG_CHANNEL_MASKS = False  # True adds 48 validation passes per fitted model
    SOURCE_NOTEBOOK_SHA256 = 'b4cfa13c837bb5e942545ad96634885f8223a034945013af76f9e7fa854e03ba'
    MODEL_VARIANT = 'amplitude_velocity'  # actual runs are specified by DIAG_EXPERIMENTS
    MODEL_MODALITY = 'both'        # 'both', 'emg', 'acc'; loader still loads both
    USE_ADAMW = False             # turn on as a separate training ablation
    LABEL_SMOOTHING = 0.0         # try 0.05 after architecture comparison
    AUGMENT_TRAIN = False         # training-only EMG gain/noise
    EMG_GAIN_STD = 0.10
    EMG_NOISE_STD = 0.01           # normalized input units
    MODEL_SEED = 42               # does not change the repetition split
    EVALUATE_TEST = False         # False: validation comparison; True: final test/refit
    RUN_TAG = (f'v7_{MODEL_VARIANT}_{MODEL_MODALITY}_seed{MODEL_SEED}'
               f'_adamw{USE_ADAMW}_ls{LABEL_SMOOTHING}_aug{AUGMENT_TRAIN}'
               f'_test{EVALUATE_TEST}')
    CKPT_DIR = KAGGLE_WORKING / f'ckpts_{RUN_TAG}'
    PLOT_DIR = KAGGLE_WORKING / f'plots_{RUN_TAG}'
    RESULTS_DIR = KAGGLE_WORKING / f'results_{RUN_TAG}'

    # Population
    INTACT_SUBJECTS = list(range(1, 21))
    AMPUTEE_SUBJECTS = [21, 22]
    SUBJECTS = list(range(1, 23))
    RUN_SUBJECTS = SUBJECTS.copy()  # use [1, 2] for a quick Kaggle smoke test
    PLOT_EACH_SUBJECT = False

    # True within-subject repetition split
    REPS_PER_GESTURE = 6
    TRAIN_REPS = 4
    VAL_REPS = 1
    TEST_REPS = 1

    # Labels 13..29 span E1 and E2
    EXERCISE_IDS = (1, 2)
    GESTURE_MIN = 13
    GESTURE_MAX = 29
    N_CLASSES = GESTURE_MAX - GESTURE_MIN + 1

    # Signal
    EMG_FS = 2000
    ACC_FS = 128  # acquisition rate from DB7 paper; verify file representation
    TARGET_FS = 2000
    EMG_KEY = 'emg'
    ACC_KEY = 'acc'
    LBL_KEY = 'restimulus'
    N_EMG_CH = 12

    # EMG + ACC multimodal experiment.
    # ACC is segmented and resampled independently inside each complete repetition.
    USE_ACC = True

    # Fixed signal preprocessing; no statistics are learned from test data.
    BANDPASS_LOW_HZ = 20.0
    BANDPASS_HIGH_HZ = 450.0
    FILTER_ORDER = 4
    NOTCH_HZ = 50.0
    NOTCH_Q = 30.0

    # Same windowing as corrected DNN/GNN
    WIN_MS = 400
    STEP_MS = 100
    TRIM_MS = 100
    WIN_SAMPLES = int(WIN_MS * TARGET_FS / 1000)
    STEP_SAMPLES = int(STEP_MS * TARGET_FS / 1000)
    TRIM_SAMPLES = int(TRIM_MS * TARGET_FS / 1000)

    # CNN
    DROPOUT = 0.15

    # Optimization
    MIN_EPOCHS = 20
    MAX_EPOCHS = 150
    PATIENCE = 15
    MIN_REFIT_EPOCHS = 10
    BATCH_SIZE = 128
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    GRAD_CLIP = 5.0
    NUM_WORKERS = 0  # safer for large in-memory subject arrays on Kaggle
    REFIT_ON_TRAIN_PLUS_VAL = False
    KEEP_SUBJECT_CHECKPOINTS = True
    CLEAN_LEGACY_V4_CACHES = False

    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if Config.DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(Config.SEED)

for directory in [
    Config.CKPT_DIR,
    Config.PLOT_DIR,
    Config.RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def _clean_known_legacy_v4_caches():
    """Delete only large caches created by the earlier v4 MKA-CNN notebook."""
    if not Config.CLEAN_LEGACY_V4_CACHES:
        return

    legacy_names = [
        'raw_continuous_cache_v4_mka_cnn_emg_acc_subject_specific_r4v1t1_strict',
        'repetition_filtered_window_cache_v4_mka_cnn_emg_acc_subject_specific_r4v1t1_strict',
    ]

    for name in legacy_names:
        path = Config.KAGGLE_WORKING / name
        if path.exists() and path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
            print(f'Removed old generated cache: {path.name}')


_clean_known_legacy_v4_caches()

assert Config.TRAIN_REPS + Config.VAL_REPS + Config.TEST_REPS == Config.REPS_PER_GESTURE
assert Config.USE_ACC is True

print(f'Device             : {Config.DEVICE}')
print(f'Population         : {Config.SUBJECTS}')
print(f'Subject models     : {Config.RUN_SUBJECTS}')
print('Evaluation         : SUBJECT-DEPENDENT / SUBJECT-SPECIFIC WITHIN-SUBJECT')
print('Training rule      : train a fresh independent CNN separately for each subject')
print('Exercise files      : E1 + E2')
print(f'Input modality     : {"EMG + ACC" if Config.USE_ACC else "EMG only"}')
print('ACC handling       : repetition-local segment + repetition-local resampling')
print(f'Window / step      : {Config.WIN_MS} / {Config.STEP_MS} ms')
print(f'Boundary trim      : {Config.TRIM_MS} ms per repetition side')
print(f'Gestures           : {Config.GESTURE_MIN}–{Config.GESTURE_MAX} ({Config.N_CLASSES} classes)')
print(f'Within split       : {Config.TRAIN_REPS} train + {Config.VAL_REPS} val + {Config.TEST_REPS} test repetitions')
print(f'Refit train+val    : {Config.REFIT_ON_TRAIN_PLUS_VAL}')
print('Persistent windows : disabled (disk-safe)')
print('Persistent raw cache: disabled (disk-safe)')
print('Actual diagnostic experiments:', Config.DIAG_EXPERIMENTS)
print('Evaluated split: VALIDATION; refit disabled; test predictions disabled.')
DIAG_RAW_AUDIT = []


In [ ]:
# Reproducibility and protocol assertions
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if Config.DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(Config.SEED)

assert Config.WIN_MS == 400
assert Config.STEP_MS == 100
assert Config.REPS_PER_GESTURE == 6
assert (Config.TRAIN_REPS, Config.VAL_REPS, Config.TEST_REPS) == (4, 1, 1)

print('Leakage-safe protocol configured.')

# Raw EMG + ACC loading — all signal transforms happen only AFTER repetition assignment


In [ ]:
class RawEMGACCPreprocessor:
    """
    Load RAW EMG, RAW accelerometer and labels from one NinaPro file.

    No filtering, no ACC resampling, no label masking and no windowing happen here.
    Keeping each source file separate preserves the exact temporal mapping needed to
    extract a matching ACC interval for every EMG repetition.
    """

    @staticmethod
    def _find_key(data, candidates):
        for candidate in candidates:
            for key in data:
                if key.lower() == candidate.lower():
                    return key
        return None

    @staticmethod
    def _time_major(array, expected_channels=None, name='signal'):
        array = np.asarray(array)

        if array.ndim == 1:
            array = array[:, None]
        if array.ndim != 2:
            raise RuntimeError(
                f'{name}: expected 2-D array, got shape {array.shape}.'
            )

        if expected_channels is not None:
            if array.shape[1] == expected_channels:
                return array
            if array.shape[0] == expected_channels:
                return array.T
            raise RuntimeError(
                f'{name}: neither dimension matches expected '
                f'{expected_channels} channels: {array.shape}.'
            )

        # For ACC the time dimension should be much larger than channel count.
        if array.shape[0] < array.shape[1] and array.shape[0] <= 128:
            array = array.T

        return array

    def apply(self, mat_path: Path):
        data = io.loadmat(str(mat_path))

        emg_key = self._find_key(data, [Config.EMG_KEY])
        acc_key = self._find_key(data, [Config.ACC_KEY])
        lbl_key = self._find_key(
            data,
            [Config.LBL_KEY, 'stimulus', 'label', 'labels'],
        )

        if emg_key is None:
            raise KeyError(f'No EMG key in {mat_path.name}.')
        if acc_key is None:
            raise KeyError(
                f'No ACC key in {mat_path.name}; EMG+ACC experiment requires accelerometer data.'
            )
        if lbl_key is None:
            raise KeyError(f'No label key in {mat_path.name}.')

        emg = self._time_major(
            data[emg_key],
            expected_channels=Config.N_EMG_CH,
            name=f'{mat_path.name} EMG',
        ).astype(np.float32)

        acc = self._time_major(
            data[acc_key],
            expected_channels=None,
            name=f'{mat_path.name} ACC',
        ).astype(np.float32)

        labels = np.asarray(
            data[lbl_key]
        ).reshape(-1).astype(np.int32)

        # restimulus is aligned with the EMG time base.
        n = min(len(emg), len(labels))
        emg = emg[:n]
        labels = labels[:n]

        if len(acc) < 2:
            raise RuntimeError(
                f'{mat_path.name}: ACC has only {len(acc)} samples.'
            )

        # Read-only structural audit. No change to preprocessing or labels.
        rep_key = self._find_key(data, ['rerepetition'])
        native = np.asarray(data[rep_key]).reshape(-1) if rep_key else None
        self.last_native_repetition = (native[:n].astype(np.int32)
            if native is not None and len(native) >= n else None)
        self.last_file_path = str(mat_path)
        unique_labels = np.unique(labels).astype(int).tolist()
        run_audit = []
        edges = np.r_[0, np.flatnonzero(np.diff(labels) != 0)+1, len(labels)]
        for start, end in zip(edges[:-1], edges[1:]):
            gesture = int(labels[start])
            if not Config.GESTURE_MIN <= gesture <= Config.GESTURE_MAX:
                continue
            ids = (np.unique(self.last_native_repetition[start:end]).astype(int).tolist()
                   if self.last_native_repetition is not None else [])
            run_audit.append(dict(gesture=gesture,start=int(start),end=int(end),
                duration_seconds=float((end-start)/Config.EMG_FS),native_rerepetition_ids=ids))
        ratio = len(acc)/float(len(emg))
        DIAG_RAW_AUDIT.append(dict(file=str(mat_path),emg_shape=list(emg.shape),
            acc_shape=list(acc.shape),original_emg_shape=list(np.shape(data[emg_key])),
            original_label_length=int(np.size(data[lbl_key])),label_key=lbl_key,
            labels_present=unique_labels,rerepetition_key=rep_key,
            rerepetition_length=int(len(native)) if native is not None else None,
            sample_count_ratio_acc_to_emg=ratio,
            implied_acc_rate_if_shared_duration=ratio*Config.EMG_FS,
            timing_interpretation=('Equal sample counts: may already be aligned; timestamps unverified.'
                if len(acc)==len(emg) else 'Unequal lengths: length-ratio mapping assumes shared start/end times; unverified.'),
            selected_gesture_runs=run_audit))
        return emg, acc, labels


class RepetitionEMGFilter:
    """Zero-phase EMG filtering applied to one already-assigned repetition only."""

    def __init__(self):
        nyquist = Config.EMG_FS / 2.0
        self.sos = butter(
            Config.FILTER_ORDER,
            [
                Config.BANDPASS_LOW_HZ / nyquist,
                Config.BANDPASS_HIGH_HZ / nyquist,
            ],
            btype='bandpass',
            output='sos',
        )
        self.b_notch, self.a_notch = iirnotch(
            Config.NOTCH_HZ / nyquist,
            Config.NOTCH_Q,
        )

    def apply(self, repetition_emg):
        repetition_emg = np.asarray(
            repetition_emg,
            dtype=np.float32,
        )

        if repetition_emg.ndim != 2:
            raise ValueError(
                f'Expected (time,channels), got {repetition_emg.shape}.'
            )
        if len(repetition_emg) < 64:
            raise RuntimeError(
                f'EMG repetition unexpectedly short: {len(repetition_emg)} samples.'
            )

        filtered = sosfiltfilt(
            self.sos,
            repetition_emg,
            axis=0,
        )
        filtered = filtfilt(
            self.b_notch,
            self.a_notch,
            filtered,
            axis=0,
        )
        return filtered.astype(np.float32)


class RepetitionACCResampler:
    """
    Map an EMG repetition interval to the corresponding interval of the RAW ACC
    recording from the SAME source file, then resample only that ACC slice.

    This is leakage-safe because resample_poly never sees ACC samples belonging
    to another train/validation/test repetition.
    """

    @staticmethod
    def extract_matching_interval(
        file_acc: np.ndarray,
        file_emg_length: int,
        emg_start: int,
        emg_end: int,
    ) -> np.ndarray:
        if not (0 <= emg_start < emg_end <= file_emg_length):
            raise ValueError(
                f'Invalid EMG interval [{emg_start}, {emg_end}) '
                f'for file length {file_emg_length}.'
            )

        ratio = len(file_acc) / float(file_emg_length)

        # ceil(start) and ceil(end) keep selected ACC timestamps inside the
        # EMG repetition's temporal interval rather than borrowing a sample
        # from the preceding repetition/rest segment.
        acc_start = int(np.ceil(emg_start * ratio))
        acc_end = int(np.ceil(emg_end * ratio))

        acc_start = max(0, min(acc_start, len(file_acc) - 1))
        acc_end = max(acc_start + 1, min(acc_end, len(file_acc)))

        acc_rep = file_acc[acc_start:acc_end].copy()
        if len(acc_rep) < 2:
            raise RuntimeError(
                f'Mapped ACC repetition is too short: {len(acc_rep)} samples.'
            )

        return acc_rep

    @staticmethod
    def resample_to_emg_length(
        acc_rep: np.ndarray,
        target_length: int,
    ) -> np.ndarray:
        source_length = len(acc_rep)
        divisor = gcd(source_length, target_length)
        up = target_length // divisor
        down = source_length // divisor

        resampled = resample_poly(
            acc_rep,
            up,
            down,
            axis=0,
        ).astype(np.float32)

        # scipy normally gives the exact target length for this reduced ratio.
        # Handle any implementation-level one-sample discrepancy without using
        # data outside this repetition.
        if len(resampled) > target_length:
            resampled = resampled[:target_length]
        elif len(resampled) < target_length:
            pad_count = target_length - len(resampled)
            pad = np.repeat(
                resampled[-1:, :],
                pad_count,
                axis=0,
            )
            resampled = np.vstack([resampled, pad])

        if len(resampled) != target_length:
            raise RuntimeError(
                f'ACC resampling length mismatch: '
                f'{len(resampled)} != {target_length}.'
            )

        return resampled.astype(np.float32)


print('Raw EMG+ACC loader, repetition-local EMG filter and ACC resampler defined.')

# Subject loader — split complete repetitions first, then process EMG + ACC locally


In [ ]:
class SubjectLoader:
    """
    Disk-safe subject loader.

    Only ONE subject is held in memory at a time. No raw signal cache and no
    overlapping-window cache is written to /kaggle/working.
    """

    def __init__(self):
        self.preprocessor = RawEMGACCPreprocessor()
        self.rep_filter = RepetitionEMGFilter()
        self.acc_resampler = RepetitionACCResampler()

    def _find_subject_dir(self, sid: int) -> Path:
        candidates = [
            Config.KAGGLE_INPUT / f'Subject_{sid}',
            Config.KAGGLE_INPUT / f'subject_{sid}',
            Config.KAGGLE_INPUT / f'S{sid}',
            Config.KAGGLE_INPUT / f's{sid}',
        ]
        for path in candidates:
            if path.is_dir():
                return path

        for path in sorted(Config.KAGGLE_INPUT.iterdir()):
            if path.is_dir() and path.name.lower().endswith(str(sid)):
                return path

        raise FileNotFoundError(
            f'Cannot find subject {sid} under {Config.KAGGLE_INPUT}.'
        )

    @staticmethod
    def _is_exercise_file(path: Path, exercise_id: int) -> bool:
        name = path.stem.upper()
        return bool(
            re.search(rf'(^|_)E{exercise_id}(_|$)', name)
        )

    def _selected_files(self, sid: int):
        subject_dir = self._find_subject_dir(sid)
        all_mat_files = (
            sorted(subject_dir.glob('*.mat'))
            or sorted(subject_dir.rglob('*.mat'))
        )

        selected = []
        for exercise_id in Config.EXERCISE_IDS:
            matches = [
                path
                for path in all_mat_files
                if self._is_exercise_file(path, exercise_id)
            ]
            if not matches:
                raise FileNotFoundError(
                    f'S{sid:02d}: missing E{exercise_id}. '
                    f'Available: {[p.name for p in all_mat_files[:15]]}'
                )
            selected.extend(matches)

        return selected

    def _load_raw_parts_from_source(self, sid: int):
        """
        Read E1/E2 directly from the read-only Kaggle input and keep them only
        for the current subject.
        """
        parts = []
        acc_channel_counts = set()

        for index, mat_file in enumerate(self._selected_files(sid)):
            emg, acc, labels = self.preprocessor.apply(mat_file)

            if emg.shape[1] != Config.N_EMG_CH:
                raise RuntimeError(
                    f'S{sid:02d}, {mat_file.name}: expected '
                    f'{Config.N_EMG_CH} EMG channels, got {emg.shape[1]}.'
                )

            if acc.shape[1] <= 0:
                raise RuntimeError(
                    f'S{sid:02d}, {mat_file.name}: ACC is required.'
                )

            acc_channel_counts.add(int(acc.shape[1]))

            parts.append({
                'file_index': int(index),
                'file_name': mat_file.name,
                'emg': emg.astype(np.float32, copy=False),
                'acc': acc.astype(np.float32, copy=False),
                'labels': labels.astype(np.int32, copy=False),
                'native_repetition': self.preprocessor.last_native_repetition,
                'source_path': self.preprocessor.last_file_path,
            })

        if len(acc_channel_counts) != 1:
            raise RuntimeError(
                f'S{sid:02d}: inconsistent ACC channel counts across E1/E2: '
                f'{sorted(acc_channel_counts)}.'
            )

        return parts, int(next(iter(acc_channel_counts)))

    @staticmethod
    def _constant_label_runs(labels: np.ndarray):
        if len(labels) == 0:
            return

        boundaries = np.flatnonzero(
            np.diff(labels) != 0
        ) + 1
        starts = np.r_[0, boundaries]
        ends = np.r_[boundaries, len(labels)]

        for start, end in zip(starts, ends):
            yield (
                int(start),
                int(end),
                int(labels[start]),
            )

    @staticmethod
    def _repetition_id(sid, gesture, repetition_index):
        return (
            int(sid),
            int(gesture),
            int(repetition_index + 1),
        )

    @staticmethod
    def _split_repetition_indices(sid, gesture):
        rng = np.random.default_rng(
            Config.SEED
            + 1009 * int(sid)
            + 9176 * int(gesture)
        )
        perm = rng.permutation(
            Config.REPS_PER_GESTURE
        ).tolist()

        test_idx = sorted(
            perm[:Config.TEST_REPS]
        )
        val_idx = sorted(
            perm[
                Config.TEST_REPS:
                Config.TEST_REPS + Config.VAL_REPS
            ]
        )
        train_idx = sorted(
            perm[
                Config.TEST_REPS
                + Config.VAL_REPS:
            ]
        )

        if (
            len(train_idx) != Config.TRAIN_REPS
            or len(val_idx) != Config.VAL_REPS
            or len(test_idx) != Config.TEST_REPS
        ):
            raise RuntimeError('Unexpected repetition split size.')

        return train_idx, val_idx, test_idx

    def _collect_repetitions(self, parts):
        runs_by_gesture = {
            gesture: []
            for gesture in range(
                Config.GESTURE_MIN,
                Config.GESTURE_MAX + 1,
            )
        }

        for part_index, part in enumerate(parts):
            for (
                run_start,
                run_end,
                gesture,
            ) in self._constant_label_runs(part['labels']):
                if gesture in runs_by_gesture:
                    runs_by_gesture[gesture].append({
                        'part_index': int(part_index),
                        'start': int(run_start),
                        'end': int(run_end),
                    })

        return runs_by_gesture

    def build_subject_splits(
        self,
        sid: int,
        expected_n_acc_ch=None,
    ):
        """
        SUBJECT-DEPENDENT / WITHIN-SUBJECT leakage-safe ordering:

        one subject's raw E1/E2
          -> identify complete repetitions
          -> 4 train / 1 val / 1 test per gesture
          -> prove repetition IDs are disjoint
          -> filter each already-assigned EMG repetition locally
          -> extract and resample matching ACC interval locally
          -> concatenate EMG+ACC
          -> trim repetition boundaries
          -> create windows inside that repetition only

        Returned arrays live in RAM only and are deleted after this subject is
        trained/evaluated.
        """
        parts, n_acc_ch = self._load_raw_parts_from_source(sid)

        if (
            expected_n_acc_ch is not None
            and int(n_acc_ch) != int(expected_n_acc_ch)
        ):
            raise RuntimeError(
                f'S{sid:02d}: ACC channels={n_acc_ch}, '
                f'expected={expected_n_acc_ch}.'
            )

        runs_by_gesture = self._collect_repetitions(parts)

        split_X = {'train': [], 'val': [], 'test': []}
        split_y = {'train': [], 'val': [], 'test': []}
        split_meta = {'train': [], 'val': [], 'test': []}
        split_rep_ids = {
            'train': set(),
            'val': set(),
            'test': set(),
        }
        assignment = {}

        expected_channels = Config.N_EMG_CH + n_acc_ch

        for gesture, runs in runs_by_gesture.items():
            if len(runs) != Config.REPS_PER_GESTURE:
                diagnostic = [
                    (
                        parts[record['part_index']]['file_name'],
                        record['start'],
                        record['end'],
                    )
                    for record in runs
                ]
                raise RuntimeError(
                    f'S{sid:02d}, gesture {gesture}: expected exactly '
                    f'{Config.REPS_PER_GESTURE} repetitions, found {len(runs)}. '
                    f'Runs={diagnostic}'
                )

            train_idx, val_idx, test_idx = (
                self._split_repetition_indices(
                    sid,
                    gesture,
                )
            )

            assignment[str(gesture)] = {
                'train_reps': [int(i + 1) for i in train_idx],
                'val_reps': [int(i + 1) for i in val_idx],
                'test_reps': [int(i + 1) for i in test_idx],
            }

            split_indices = {
                'train': train_idx,
                'val': val_idx,
                'test': test_idx,
            }
            class_id = gesture - Config.GESTURE_MIN

            # Register identities BEFORE filtering, resampling, or windowing.
            for split_name, rep_indices in split_indices.items():
                for rep_index in rep_indices:
                    split_rep_ids[split_name].add(
                        self._repetition_id(
                            sid,
                            gesture,
                            rep_index,
                        )
                    )

            assert split_rep_ids['train'].isdisjoint(
                split_rep_ids['val']
            )
            assert split_rep_ids['train'].isdisjoint(
                split_rep_ids['test']
            )
            assert split_rep_ids['val'].isdisjoint(
                split_rep_ids['test']
            )

            for split_name, rep_indices in split_indices.items():
                for rep_index in rep_indices:
                    record = runs[rep_index]
                    part = parts[record['part_index']]

                    run_start = record['start']
                    run_end = record['end']
                    native_ids = (np.unique(part['native_repetition'][run_start:run_end]).tolist()
                                  if part['native_repetition'] is not None else [])

                    # RAW EMG repetition.
                    emg_raw = part['emg'][
                        run_start:run_end
                    ].copy()

                    # Matching RAW ACC interval from the SAME source file.
                    acc_raw = (
                        self.acc_resampler.extract_matching_interval(
                            file_acc=part['acc'],
                            file_emg_length=len(part['emg']),
                            emg_start=run_start,
                            emg_end=run_end,
                        )
                    )

                    # Both operations see ONLY this already-assigned repetition.
                    emg_filtered = self.rep_filter.apply(
                        emg_raw
                    )
                    acc_resampled = (
                        self.acc_resampler.resample_to_emg_length(
                            acc_raw,
                            target_length=len(emg_filtered),
                        )
                    )

                    if acc_resampled.shape[1] != n_acc_ch:
                        raise RuntimeError(
                            f'S{sid:02d}, gesture {gesture}, rep '
                            f'{rep_index + 1}: ACC channel mismatch.'
                        )

                    repetition_signal = np.concatenate(
                        [
                            emg_filtered,
                            acc_resampled,
                        ],
                        axis=1,
                    ).astype(np.float32)

                    if repetition_signal.shape[1] != expected_channels:
                        raise RuntimeError(
                            f'S{sid:02d}: expected {expected_channels} '
                            f'combined channels, got '
                            f'{repetition_signal.shape[1]}.'
                        )

                    clean_start = Config.TRIM_SAMPLES
                    clean_end = (
                        len(repetition_signal)
                        - Config.TRIM_SAMPLES
                    )

                    if (
                        clean_end - clean_start
                        < Config.WIN_SAMPLES
                    ):
                        raise RuntimeError(
                            f'S{sid:02d}, gesture {gesture}, rep '
                            f'{rep_index + 1}: too short after trim.'
                        )

                    clean_rep = repetition_signal[
                        clean_start:clean_end
                    ]

                    for start in range(
                        0,
                        len(clean_rep)
                        - Config.WIN_SAMPLES
                        + 1,
                        Config.STEP_SAMPLES,
                    ):
                        window = clean_rep[
                            start:
                            start + Config.WIN_SAMPLES
                        ].T.copy()

                        split_X[split_name].append(window)
                        split_y[split_name].append(class_id)
                        split_meta[split_name].append(dict(subject=sid,split=split_name,
                            file_name=part['file_name'],source_path=part['source_path'],gesture=gesture,
                            run_index=int(rep_index+1),run_start_emg_sample=int(run_start),
                            run_end_emg_sample=int(run_end),
                            window_start_emg_sample=int(run_start+clean_start+start),
                            window_end_emg_sample=int(run_start+clean_start+start+Config.WIN_SAMPLES),
                            native_rerepetition_ids=','.join(str(int(v)) for v in native_ids)))

                    del (
                        emg_raw,
                        acc_raw,
                        emg_filtered,
                        acc_resampled,
                        repetition_signal,
                        clean_rep,
                    )

        # Final repetition-level leakage proof.
        assert split_rep_ids['train'].isdisjoint(
            split_rep_ids['val']
        )
        assert split_rep_ids['train'].isdisjoint(
            split_rep_ids['test']
        )
        assert split_rep_ids['val'].isdisjoint(
            split_rep_ids['test']
        )

        expected_total_reps = (
            Config.N_CLASSES
            * Config.REPS_PER_GESTURE
        )
        all_rep_ids = (
            split_rep_ids['train']
            | split_rep_ids['val']
            | split_rep_ids['test']
        )

        if len(all_rep_ids) != expected_total_reps:
            raise RuntimeError(
                f'S{sid:02d}: expected {expected_total_reps} unique '
                f'repetition IDs, found {len(all_rep_ids)}.'
            )

        result = {}

        for split_name in ('train', 'val', 'test'):
            if not split_X[split_name]:
                raise RuntimeError(
                    f'S{sid:02d}: no {split_name} windows created.'
                )

            X = np.stack(
                split_X[split_name]
            ).astype(np.float32)
            y = np.asarray(
                split_y[split_name],
                dtype=np.int64,
            )

            if X.shape[1] != expected_channels:
                raise RuntimeError(
                    f'S{sid:02d}: {split_name} channel mismatch.'
                )

            present = set(np.unique(y).tolist())
            expected_classes = set(
                range(Config.N_CLASSES)
            )
            if present != expected_classes:
                missing = sorted(
                    expected_classes - present
                )
                raise RuntimeError(
                    f'S{sid:02d}: {split_name} missing '
                    f'class IDs {missing}.'
                )

            result[f'X_{split_name}'] = X
            result[f'y_{split_name}'] = y
            result[f'meta_{split_name}'] = split_meta[split_name]
            assert len(split_meta[split_name]) == len(y)

        provenance = {
            split_name: [
                list(rep_id)
                for rep_id in sorted(
                    split_rep_ids[split_name]
                )
            ]
            for split_name in ('train', 'val', 'test')
        }

        print(
            f'  S{sid:02d}: RAM-only strict EMG+ACC windows | '
            f'train={result["X_train"].shape}, '
            f'val={result["X_val"].shape}, '
            f'test={result["X_test"].shape}'
        )
        print(
            f'    {expected_channels} channels '
            f'({Config.N_EMG_CH} EMG + {n_acc_ch} ACC) | '
            f'repetition IDs '
            f'{len(split_rep_ids["train"])}/'
            f'{len(split_rep_ids["val"])}/'
            f'{len(split_rep_ids["test"])} | disjoint'
        )

        del parts, split_X, split_y
        gc.collect()

        return result, assignment, provenance, n_acc_ch


print(
    'Disk-safe subject-dependent leakage-safe '
    'EMG+ACC SubjectLoader defined.'
)

# Training-only channel normalization


In [ ]:
class ChannelNormalizer:
    """Per-channel mean/std. Fit only on the current subject's training pool."""

    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, X: np.ndarray):
        if len(X) == 0:
            raise ValueError('Cannot fit ChannelNormalizer on an empty array.')
        self.mean = X.mean(axis=(0, 2), keepdims=True)
        self.std = X.std(axis=(0, 2), keepdims=True) + 1e-8
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.mean is None or self.std is None:
            raise RuntimeError('ChannelNormalizer must be fitted before transform().')
        return ((X - self.mean) / self.std).astype(np.float32)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


print('Training-only ChannelNormalizer defined.')

# PyTorch dataset


In [ ]:
class EMGWindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.y = torch.from_numpy(y.astype(np.int64, copy=False))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Multi-Kernel Attention 1D CNN


In [ ]:
import torch
import torch.nn as nn


class ParallelMultiKernelBlock(nn.Module):
    """
    True multi-kernel block: parallel branches with different kernel sizes
    (default 3, 5, 7) processed at the SAME depth and concatenated along the
    channel dimension, then merged with a 1x1 conv. This captures multi-scale
    temporal patterns simultaneously (unlike a sequential 7->5->3 design,
    which only changes kernel size across depth, not within one stage).
    """

    def __init__(
        self,
        in_ch,
        out_ch,
        kernels=(3, 5, 7),
        pool=True,
        dropout=0.1,
    ):
        super().__init__()

        for k in kernels:
            assert k % 2 == 1, (
                f"kernel size {k} must be odd so that "
                f"padding=kernel//2 gives symmetric 'same' padding"
            )

        branch_sizes = self._split_channels(out_ch, len(kernels))

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, b_ch, k, padding=k // 2, bias=False),
                nn.BatchNorm1d(b_ch),
                nn.ReLU(inplace=True),
                nn.Conv1d(b_ch, b_ch, k, padding=k // 2, bias=False),
                nn.BatchNorm1d(b_ch),
                nn.ReLU(inplace=True),
            )
            for k, b_ch in zip(kernels, branch_sizes)
        ])

        self.merge = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )

        # Dropout1d zeroes whole feature channels (not individual elements) —
        # correct form of regularization for conv feature maps.
        self.drop = nn.Dropout1d(dropout) if dropout > 0 else nn.Identity()
        self.pool = nn.MaxPool1d(2) if pool else nn.Identity()

    @staticmethod
    def _split_channels(total, n):
        # Splits `total` channels as evenly as possible across `n` branches
        # so the concatenated output is exactly `out_ch` (e.g. 64 -> [22,21,21]).
        base, rem = divmod(total, n)
        return [base + 1 if i < rem else base for i in range(n)]

    def forward(self, x):
        x = torch.cat([branch(x) for branch in self.branches], dim=1)
        x = self.merge(x)
        x = self.drop(x)
        return self.pool(x)


class ChannelAttentionBlock(nn.Module):
    """
    Squeeze-and-Excitation style CHANNEL attention. Learns which learned feature channels
    matter most; it does NOT attend across time
    steps. Named explicitly so it isn't confused with temporal attention.
    """

    def __init__(self, in_ch, reduction=4):
        super().__init__()
        reduced = max(in_ch // reduction, 1)

        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(in_ch, reduced, 1),
            nn.ReLU(inplace=True),
            nn.Conv1d(reduced, in_ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.attention(x)


class TemporalAttentionPool(nn.Module):
    """
    Learned attention-weighted pooling over the time axis, replacing plain
    global average pooling. A 1x1 conv scores every time step, softmax turns
    scores into weights, and the weighted sum replaces a uniform mean — so
    the model can down-weight uninformative (e.g. resting) time steps instead
    of averaging them in blindly.
    """

    def __init__(self, in_ch):
        super().__init__()
        self.score = nn.Conv1d(in_ch, 1, kernel_size=1)

    def forward(self, x):
        # x: (batch, channels, time)
        weights = torch.softmax(self.score(x), dim=-1)   # (batch, 1, time)
        return (x * weights).sum(dim=-1)                 # (batch, channels)


class OriginalMultiKernelAttention1DCNN(nn.Module):
    def __init__(
        self,
        n_channels,
        n_classes,
        dropout=0.15,
    ):
        super().__init__()
        self.n_channels = int(n_channels)

        self.stage1 = ParallelMultiKernelBlock(
            n_channels, 64, kernels=(3, 5, 7), pool=True, dropout=dropout
        )
        self.attn1 = ChannelAttentionBlock(64)

        self.stage2 = ParallelMultiKernelBlock(
            64, 128, kernels=(3, 5, 7), pool=True, dropout=dropout
        )
        self.attn2 = ChannelAttentionBlock(128)

        self.stage3 = ParallelMultiKernelBlock(
            128, 256, kernels=(3, 5, 7), pool=False, dropout=dropout
        )
        self.attn3 = ChannelAttentionBlock(256)

        self.temporal_pool = TemporalAttentionPool(256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.attn1(x)
        x = self.stage2(x)
        x = self.attn2(x)
        x = self.stage3(x)
        x = self.attn3(x)
        x = self.temporal_pool(x)
        return self.head(x)

    def count_params(self):
        return sum(
            parameter.numel()
            for parameter in self.parameters()
            if parameter.requires_grad
        )



import torch.nn.functional as F



"""Paper-inspired 1D amplitude/velocity ablation; not a T-EKIM implementation."""

def derivative_statistics(X, fs=2000.0, n_emg=12, batch_size=128):
    """Fit only on normalized TRAIN windows; omit padded boundary positions.

    X has shape (windows, original channels, time). Derivatives are computed
    independently inside each window, so no repetition boundary is crossed.
    Overlapping training windows contribute repeated samples, as in the existing
    channel normalizer. No validation or test array is accepted by the runner.
    """
    if X.ndim != 3 or len(X) == 0 or X.shape[1] < n_emg or X.shape[2] < 2:
        raise ValueError('Expected nonempty (windows, channels, time>=2).')
    if fs <= 0:
        raise ValueError('Sampling rate must be positive.')
    total = np.zeros(n_emg, dtype=np.float64)
    total_sq = np.zeros(n_emg, dtype=np.float64)
    count = 0
    for start in range(0, len(X), batch_size):
        chunk = X[start:start+batch_size, :n_emg].astype(np.float64)
        derivative = np.diff(chunk, axis=-1) * fs
        if not np.isfinite(derivative).all():
            raise FloatingPointError('Nonfinite training derivative.')
        total += derivative.sum(axis=(0, 2))
        total_sq += np.square(derivative).sum(axis=(0, 2))
        count += derivative.shape[0] * derivative.shape[2]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq/count - mean**2, 0))
    return mean.astype(np.float32), np.maximum(std, 1e-6).astype(np.float32)


class MultiKernelAttention1DCNN(nn.Module):
    """Original multi-kernel CNN with optional 12 standardized EMG derivatives.

    Input remains B x 48 x 800 (12 EMG, then 36 ACC); the candidate appends
    12 derivatives internally, giving B x 60 x 800. The original convolution
    blocks, kernels, widths, attention, pooling and head are unchanged.
    Constructor metadata + state_dict + original normalizer suffice to reload.
    """
    def __init__(self, n_channels, n_classes, dropout=0.15,
                 variant='baseline', modality='both', n_emg=12, fs=2000.0):
        super().__init__()
        if variant not in ('baseline', 'amplitude_velocity'):
            raise ValueError('Use baseline or amplitude_velocity.')
        if modality != 'both':
            raise ValueError('This controlled comparison keeps both EMG and ACC.')
        if n_channels <= n_emg:
            raise ValueError('Expected EMG followed by ACC channels.')
        self.n_channels = int(n_channels)
        self.n_emg = int(n_emg)
        self.fs = float(fs)
        self.variant, self.modality = variant, modality
        self.register_buffer('derivative_mean', torch.zeros(1, n_emg, 1))
        self.register_buffer('derivative_std', torch.ones(1, n_emg, 1))
        self.register_buffer('derivative_fitted', torch.tensor(False))
        self.network = OriginalMultiKernelAttention1DCNN(
            n_channels + (n_emg if variant == 'amplitude_velocity' else 0),
            n_classes, dropout)

    def fit_preprocessor(self, X_train):
        if self.variant == 'amplitude_velocity':
            mean, std = derivative_statistics(X_train, self.fs, self.n_emg)
            with torch.no_grad():
                self.derivative_mean.copy_(torch.as_tensor(mean).reshape(1, -1, 1))
                self.derivative_std.copy_(torch.as_tensor(std).reshape(1, -1, 1))
                self.derivative_fitted.fill_(True)
        return self

    def prepare_input(self, x, mask_derivative=False):
        if x.ndim != 3 or x.shape[1] != self.n_channels or x.shape[-1] < 2:
            raise ValueError('Expected (batch, original channels, time>=2).')
        if self.variant == 'baseline':
            return x
        if not self.derivative_fitted.item():
            raise RuntimeError('Fit derivative statistics on training data or load a fitted checkpoint first.')
        derivative = (x[:, :self.n_emg, 1:] - x[:, :self.n_emg, :-1]) * self.fs
        derivative = (derivative-self.derivative_mean) / self.derivative_std
        # No preceding sample exists at t=0. Pad with zero in standardized units.
        derivative = F.pad(derivative, (1, 0), value=0.0)
        if mask_derivative:
            derivative = torch.zeros_like(derivative)
        return torch.cat([x, derivative], dim=1)

    def forward(self, x, mask_derivative=False):
        return self.network(self.prepare_input(x, mask_derivative))

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def save_derivative_health(model, X_train, X_val, directory):
    """Read-only summary of the fitted transform; never refits on validation."""
    if model.variant != 'amplitude_velocity':
        return
    means=model.derivative_mean.detach().cpu().numpy().reshape(-1)
    stds=model.derivative_std.detach().cpu().numpy().reshape(-1)
    rows=[]
    for split,X in [('train',X_train),('validation',X_val)]:
        ids=np.linspace(0,len(X)-1,min(128,len(X)),dtype=int)
        d=np.diff(X[ids,:model.n_emg].astype(np.float64),axis=-1)*model.fs
        z=(d-means[None,:,None])/stds[None,:,None]
        for ch in range(model.n_emg):
            rows.append(dict(split=split,emg_channel=ch,sampled_windows=len(ids),
                fitted_derivative_mean=float(means[ch]),fitted_derivative_std=float(stds[ch]),
                standardized_derivative_mean=float(z[:,ch].mean()),
                standardized_derivative_std=float(z[:,ch].std()),
                extreme_fraction_abs_gt_10=float((np.abs(z[:,ch])>10).mean())))
    pd.DataFrame(rows).to_csv(directory/'derivative_health.csv',index=False)
    np.savez_compressed(directory/'derivative_normalizer.npz',mean=means,std=stds,fs=model.fs)


In [ ]:
# Preprocessing and checkpoint checks before subject training.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(7)
    sample=np.random.default_rng(7).normal(size=(4,48,800)).astype(np.float32)
    for variant in ('baseline','amplitude_velocity'):
        net=MultiKernelAttention1DCNN(48,17,variant=variant).fit_preprocessor(sample)
        x=torch.from_numpy(sample[:2])
        prepared=net.prepare_input(x)
        assert prepared.shape==(2,60 if variant=='amplitude_velocity' else 48,800)
        assert torch.equal(prepared[:,:48],x)
        logits=net(x)
        assert logits.shape==(2,17) and torch.isfinite(logits).all()
        nn.CrossEntropyLoss()(logits,torch.tensor([0,16])).backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in net.parameters())
        if variant=='amplitude_velocity':
            assert torch.count_nonzero(prepared[:,48:,0])==0
            assert torch.count_nonzero(net.prepare_input(x,True)[:,48:])==0
            mean=net.derivative_mean.clone()
            net(x*2)
            assert torch.equal(mean,net.derivative_mean), 'Inference changed fitted statistics'
        net.eval()
        clone=MultiKernelAttention1DCNN(48,17,variant=variant)
        clone.load_state_dict(net.state_dict()); clone.eval()
        with torch.no_grad(): assert torch.equal(net(x),clone(x))
        print(variant,net.count_params(),'preprocessing / gradient / reload PASS')
        del net,clone,x,prepared,logits


# Compute GFLOPs


In [ ]:
def compute_gflops(model, n_channels, win_samples=Config.WIN_SAMPLES):
    if HAS_THOP:
        dummy = torch.zeros(1, n_channels, win_samples)
        macs, _ = thop_profile(deepcopy(model).cpu(), inputs=(dummy,), verbose=False)
        return macs * 2 / 1e9
    else:
        return -1.0

# Trainer — validation selects epoch; final model can be refitted on train+validation


In [ ]:
class Trainer:
    def __init__(self, model, save_path: Path, n_classes: int):
        self.model = model.to(Config.DEVICE)
        self.save_path = save_path
        self.n_classes = n_classes
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': [],
        }
        self.best_epoch = 0
        self.best_val_loss = float('inf')
        self.train_wall = 0.0

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)

        total_loss, correct, total = 0.0, 0, 0
        context = torch.enable_grad() if training else torch.no_grad()

        with context:
            for X, y in loader:
                X = X.to(Config.DEVICE, non_blocking=True)
                y = y.to(Config.DEVICE, non_blocking=True)

                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(
                        emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = (emg * gain
                        + Config.EMG_NOISE_STD * torch.randn_like(emg))
                logits = self.model(X)
                loss = criterion(logits, y)

                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(),
                        Config.GRAD_CLIP,
                    )
                    optimizer.step()

                total_loss += loss.item() * len(y)
                correct += (logits.argmax(1) == y).sum().item()
                total += len(y)

        return total_loss / max(total, 1), correct / max(total, 1)

    def fit(self, train_loader, val_loader):
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(
            self.model.parameters(),
            lr=Config.LR,
            weight_decay=Config.WEIGHT_DECAY,
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=Config.MAX_EPOCHS,
        )

        patience_count = 0
        start_wall = time.perf_counter()

        for epoch in range(1, Config.MAX_EPOCHS + 1):
            tr_loss, tr_acc = self._run_epoch(
                train_loader, optimizer, criterion
            )
            vl_loss, vl_acc = self._run_epoch(
                val_loader, criterion=criterion
            )
            scheduler.step()

            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)

            if vl_loss < self.best_val_loss:
                self.best_val_loss = float(vl_loss)
                self.best_epoch = int(epoch)
                patience_count = 0
                torch.save(self.model.state_dict(), self.save_path)
            else:
                patience_count += 1

            if epoch == 1 or epoch % 10 == 0:
                print(
                    f'  Epoch {epoch:3d} | '
                    f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
                    f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}'
                )

            if epoch >= Config.MIN_EPOCHS and patience_count >= Config.PATIENCE:
                print(
                    f'  Early stop at epoch {epoch} '
                    f'(patience={Config.PATIENCE})'
                )
                break

        self.train_wall = time.perf_counter() - start_wall
        print(
            f'  Selection training: {self.train_wall:.1f} s | '
            f'best epoch={self.best_epoch} | '
            f'best val_loss={self.best_val_loss:.4f}'
        )
        return self

    def fit_fixed_epochs(self, train_loader, epochs: int):
        """Fresh final model training on train+validation after epoch selection."""
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(
            self.model.parameters(),
            lr=Config.LR,
            weight_decay=Config.WEIGHT_DECAY,
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=Config.MAX_EPOCHS  # preserve the selection LR trajectory,
        )

        start_wall = time.perf_counter()
        for epoch in range(1, int(epochs) + 1):
            loss, acc = self._run_epoch(
                train_loader, optimizer, criterion
            )
            scheduler.step()

            if epoch == 1 or epoch % 10 == 0 or epoch == int(epochs):
                print(
                    f'  Refit epoch {epoch:3d}/{int(epochs)} | '
                    f'loss={loss:.4f} | acc={acc:.4f}'
                )

        self.train_wall = time.perf_counter() - start_wall
        torch.save(self.model.state_dict(), self.save_path)
        print(f'  Refit training: {self.train_wall:.1f} s')
        return self


print('Trainer defined.')

# Diagnostic run
Outputs are saved incrementally. A caught exception writes FAILURE.txt and bundles completed outputs.
A hard kernel kill can bypass ZIP creation; the saved files still appear in Kaggle Output.


In [ ]:
"""Diagnostic runner embedded in the delivered Kaggle notebook."""
import sys
import platform
import traceback
import hashlib
import zipfile
from datetime import datetime, timezone
from sklearn.metrics import classification_report, balanced_accuracy_score
from scipy.signal import welch


def json_write(path, obj):
    def convert(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, np.generic):
            return x.item()
        return str(x)
    Path(path).write_text(json.dumps(obj, indent=2, default=convert), encoding='utf-8')


def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)


def probability_metrics(y, p):
    """Uncalibrated confidence diagnostics; multiclass Brier is a sum over classes."""
    pred = p.argmax(1)
    confidence = p.max(1)
    correct = pred == y
    bins = np.minimum((confidence * 10).astype(int), 9)
    calibration = []
    ece = 0.0
    for b in range(10):
        mask = bins == b
        n = int(mask.sum())
        acc = float(correct[mask].mean()) if n else None
        conf = float(confidence[mask].mean()) if n else None
        calibration.append(dict(bin=b, lower=b/10, upper=(b+1)/10,
                                count=n, accuracy=acc, confidence=conf))
        if n:
            ece += n / len(y) * abs(acc - conf)
    targets = np.eye(p.shape[1])[y]
    result = dict(
        accuracy=float(correct.mean()),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)),
        f1_macro=float(f1_score(y, pred, labels=np.arange(p.shape[1]),
                              average='macro', zero_division=0)),
        f1_weighted=float(f1_score(y, pred, average='weighted', zero_division=0)),
        nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()),
        brier_multiclass=float(((p - targets)**2).sum(1).mean()),
        ece_10_bins=float(ece),
        high_confidence_error_fraction=float(((confidence >= .9) & ~correct).mean()),
        n_windows=int(len(y)))
    return result, calibration


def predict_arrays(model, X, mask=None):
    """Order-preserving prediction, optional zero-at-training-mean stress test."""
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(X), Config.BATCH_SIZE):
            x = torch.as_tensor(X[start:start+Config.BATCH_SIZE], device=Config.DEVICE)
            if mask is not None:
                x = x.clone()
                if mask == 'emg':
                    x[:, :Config.N_EMG_CH] = 0
                elif mask == 'acc':
                    x[:, Config.N_EMG_CH:] = 0
                elif mask != 'derivative':
                    x[:, int(mask)] = 0
            p = torch.softmax(model(x, mask_derivative=(mask == 'derivative')), dim=1)
            if not torch.isfinite(p).all():
                raise FloatingPointError('Non-finite predictions')
            out.append(p.cpu().numpy())
    return np.concatenate(out)


class DiagnosticTrainer(Trainer):
    """Preserves Trainer.fit selection logic; records each epoch without extra forwards."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.epoch_rows = []
        self.last_training = {}

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)
        total_loss, correct, total = 0.0, 0, 0
        norms, pred_all, y_all = [], [], []
        tick = time.perf_counter()
        with torch.set_grad_enabled(training):
            for X, y in loader:
                X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(
                        emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = emg * gain + Config.EMG_NOISE_STD * torch.randn_like(emg)
                logits = self.model(X)
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite loss; inspect signal_health.csv')
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    if not torch.isfinite(norm):
                        raise FloatingPointError('Non-finite gradient norm')
                    norms.append(float(norm.item()))
                    optimizer.step()
                pred = logits.argmax(1)
                total_loss += loss.item() * len(y)
                correct += (pred == y).sum().item()
                total += len(y)
                pred_all.extend(pred.detach().cpu().tolist())
                y_all.extend(y.cpu().tolist())
        values = dict(loss=total_loss/total, accuracy=correct/total,
                      f1_macro=float(f1_score(y_all, pred_all, labels=range(self.n_classes),
                                             average='macro', zero_division=0)),
                      seconds=time.perf_counter()-tick)
        if training:
            self.last_training = {'train_'+k:v for k,v in values.items()}
            self.last_training.update(lr=optimizer.param_groups[0]['lr'],
                gradient_norm_mean=float(np.mean(norms)), gradient_norm_max=float(np.max(norms)),
                gradient_clipped_fraction=float(np.mean(np.array(norms)>Config.GRAD_CLIP)))
        else:
            row = dict(epoch=len(self.epoch_rows)+1, **self.last_training,
                       **{'val_'+k:v for k,v in values.items()})
            self.epoch_rows.append(row)
            # Persist after every epoch: useful even if Kaggle interrupts later.
            pd.DataFrame(self.epoch_rows).to_csv(self.save_path.parent/'history.csv', index=False)
        return values['loss'], values['accuracy']


def signal_diagnostics(data, normalizer, directory):
    """Train/validation only. Samples windows to bound temporary memory."""
    rows = []
    for split in ('train', 'val'):
        X = data['X_'+split]
        ids = np.linspace(0, len(X)-1, min(len(X), 128), dtype=int)
        sample = X[ids]
        for ch in range(X.shape[1]):
            x = sample[:, ch, :].astype(np.float64)
            finite = np.isfinite(x)
            good = x[finite]
            mu = float(normalizer.mean[0, ch, 0])
            sd = float(normalizer.std[0, ch, 0])
            rows.append(dict(split=split, channel=ch,
                modality='emg' if ch < Config.N_EMG_CH else 'acc',
                sampled_windows=len(ids), nonfinite_fraction=float(1-finite.mean()),
                mean=float(good.mean()) if good.size else None,
                std=float(good.std()) if good.size else None,
                normalized_mean=float((good.mean()-mu)/sd) if good.size else None,
                normalized_std=float(good.std()/sd) if good.size else None,
                normalized_extreme_fraction=float((np.abs((good-mu)/sd)>10).mean()) if good.size else None,
                constant_within_window_fraction=float((np.ptp(x, axis=1)<1e-12).mean())))
    stats = pd.DataFrame(rows)
    stats.to_csv(directory/'signal_health.csv', index=False)
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    for split in ('train', 'val'):
        part = stats[stats.split == split]
        axes[0].plot(part.channel, part.normalized_mean, label=split)
        axes[1].plot(part.channel, part.normalized_std, label=split)
    for ax in axes:
        ax.axvline(11.5, color='gray', linestyle='--')
        ax.legend()
        ax.set_xlabel('Input channel: EMG 0–11; ACC 12–47')
    axes[0].set_ylabel('Mean in training SD units')
    axes[1].set_ylabel('SD / training SD')
    save_figure(fig, directory/'channel_shift.png')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    spectral = []
    for split in ('train', 'val'):
        X = data['X_'+split]
        ids = np.linspace(0, len(X)-1, min(24, len(X)), dtype=int)
        for ax, (name, channel_slice) in zip(axes, [('emg', slice(0,12)), ('acc', slice(12,None))]):
            f, p = welch(X[ids, channel_slice], fs=Config.TARGET_FS,
                         nperseg=min(512, X.shape[-1]), axis=-1)
            power = p.mean(axis=(0,1))
            ax.semilogy(f, power+1e-20, label=split)
            spectral.extend(dict(split=split, modality=name, frequency_hz=float(a), power=float(b))
                            for a,b in zip(f,power))
            ax.set_xlim(0, 500 if name=='emg' else 100)
            ax.set_title(name.upper()+' after existing preprocessing')
            ax.set_xlabel('Hz'); ax.legend()
    save_figure(fig, directory/'signal_spectrum.png')
    pd.DataFrame(spectral).to_csv(directory/'signal_spectrum.csv', index=False)
    return stats


def integrity_report(data, directory):
    """Check actual sample intervals and available native IDs, not just synthetic IDs."""
    runs={}
    native={}
    issues=[]
    for split in ('train','val','test'):
        for record in data['meta_'+split]:
            key=(record['source_path'],record['gesture'],record['run_index'])
            runs[key]=record
            if not (record['run_start_emg_sample'] <= record['window_start_emg_sample']
                    < record['window_end_emg_sample'] <= record['run_end_emg_sample']):
                issues.append(dict(severity='error',issue='Window outside its repetition',record=record))
            if record['window_end_emg_sample']-record['window_start_emg_sample'] != Config.WIN_SAMPLES:
                issues.append(dict(severity='error',issue='Wrong window length',record=record))
        assert len(data['meta_'+split])==len(data['y_'+split])
        expected=np.array([r['gesture']-Config.GESTURE_MIN for r in data['meta_'+split]])
        if not np.array_equal(expected,data['y_'+split]):
            issues.append(dict(severity='error',issue='Metadata and labels not aligned',split=split))
    records=list(runs.values())
    for r in records:
        ids=[int(v) for v in r['native_rerepetition_ids'].split(',') if v]
        if len(ids)!=1 or ids[0] not in range(1,Config.REPS_PER_GESTURE+1):
            issues.append(dict(severity='review',issue='Missing, mixed or unexpected native rerepetition ID',
                               file=r['file_name'],gesture=r['gesture'],run_index=r['run_index'],native_ids=ids))
        else:
            key=(r['source_path'],r['gesture'],ids[0])
            native.setdefault(key,set()).add(r['split'])
    for key,splits in native.items():
        if len(splits)>1:
            issues.append(dict(severity='error',issue='Same native repetition assigned to multiple splits',
                               native_key=list(key),splits=sorted(splits)))
    for i,a in enumerate(records):
        for b in records[i+1:]:
            if a['source_path']==b['source_path'] and a['split']!=b['split']:
                if max(a['run_start_emg_sample'],b['run_start_emg_sample']) < min(a['run_end_emg_sample'],b['run_end_emg_sample']):
                    issues.append(dict(severity='error',issue='Cross-split source interval overlap',first=a,second=b))
    json_write(directory/'integrity_checks.json',dict(unique_runs=len(records),
        windows={s:len(data['y_'+s]) for s in ('train','val','test')},issues=issues,
        interpretation='No listed issue means these checks passed, not proof of all data assumptions.'))
    if any(i['severity']=='error' for i in issues):
        raise RuntimeError('Data-integrity check failed; inspect integrity_checks.json before training.')


def prediction_report(model, X, y, metadata, directory, split):
    p = predict_arrays(model, X)
    metrics, calibration = probability_metrics(y, p)
    pred, conf = p.argmax(1), p.max(1)
    frame = pd.DataFrame(metadata).copy()
    assert len(frame) == len(y)
    frame['true_class'] = y
    frame['predicted_class'] = pred
    frame['predicted_gesture'] = pred + Config.GESTURE_MIN
    frame['correct'] = pred == y
    frame['confidence'] = conf
    frame['true_class_probability'] = p[np.arange(len(y)), y]
    top3 = np.argsort(-p, axis=1)[:, :3]
    for rank in range(3):
        frame[f'top{rank+1}_gesture'] = top3[:,rank]+Config.GESTURE_MIN
    frame.to_csv(directory/f'{split}_predictions.csv', index=False)
    np.savez_compressed(directory/f'{split}_probabilities.npz', y_true=y, probabilities=p)
    frame[~frame.correct].sort_values('confidence', ascending=False).head(100).to_csv(
        directory/f'{split}_confident_errors.csv', index=False)
    cm = confusion_matrix(y,pred,labels=np.arange(Config.N_CLASSES))
    class_names = [f'G{i}' for i in range(Config.GESTURE_MIN,Config.GESTURE_MAX+1)]
    pd.DataFrame(cm,index=class_names,columns=class_names).to_csv(directory/f'{split}_confusion_counts.csv')
    report = classification_report(y,pred,labels=np.arange(Config.N_CLASSES),
        target_names=class_names,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(directory/f'{split}_classification_report.csv')
    if split == 'validation':
        cmn = cm / np.maximum(cm.sum(1,keepdims=True),1)
        fig, ax = plt.subplots(figsize=(11,9))
        sns.heatmap(cmn,ax=ax,vmin=0,vmax=1,cmap='Blues',annot=True,fmt='.2f',
                    xticklabels=class_names,yticklabels=class_names)
        ax.set(xlabel='Predicted gesture',ylabel='True gesture',title='Validation recall by gesture')
        save_figure(fig,directory/'validation_confusion.png')
        pairs = [(class_names[i],class_names[j],int(cm[i,j]),float(cmn[i,j]))
                 for i in range(len(cm)) for j in range(len(cm)) if i!=j and cm[i,j]]
        pd.DataFrame(sorted(pairs,key=lambda z:-z[2]),columns=['true','predicted','count','fraction_of_true_class']).to_csv(
            directory/'confusion_pairs.csv',index=False)
        cal = pd.DataFrame(calibration)
        cal.to_csv(directory/'calibration.csv',index=False)
        fig, axes = plt.subplots(1,2,figsize=(10,4))
        active = cal[cal['count']>0]
        axes[0].plot(active.confidence,active.accuracy,'o-')
        axes[0].plot([0,1],[0,1],'--',color='gray')
        axes[0].set(xlabel='Mean confidence',ylabel='Observed accuracy',title='Reliability (uncalibrated)')
        for correct,label in [(True,'Correct'),(False,'Wrong')]:
            axes[1].hist(conf[(pred==y)==correct],bins=np.linspace(0,1,11),alpha=.6,label=label)
        axes[1].set(xlabel='Confidence',ylabel='Window count'); axes[1].legend()
        save_figure(fig,directory/'confidence.png')
        fig, axes = plt.subplots(2,1,figsize=(13,5),sharex=True)
        axes[0].plot(frame.gesture.to_numpy(),'.',label='True')
        axes[0].plot(frame.predicted_gesture.to_numpy(),'.',alpha=.5,label='Predicted')
        axes[0].legend(); axes[0].set_ylabel('Gesture ID')
        axes[1].plot(conf); axes[1].set(xlabel='Window index (class/repetition order; not continuous time)',ylabel='Confidence')
        save_figure(fig,directory/'validation_prediction_sequence.png')
    # These are within-repetition window summaries, not additional independent trials.
    frame.groupby(['file_name','gesture','run_index'],dropna=False).agg(
        windows=('correct','size'),window_accuracy=('correct','mean'),
        mean_confidence=('confidence','mean')).reset_index().to_csv(directory/f'{split}_repetition_summary.csv',index=False)
    json_write(directory/f'{split}_metrics.json',metrics)
    return metrics,p


def learning_report(trainer, directory):
    h = pd.DataFrame(trainer.epoch_rows)
    h.to_csv(directory/'history.csv',index=False)
    fig, axes = plt.subplots(2,2,figsize=(12,8))
    for split in ('train','val'):
        axes[0,0].plot(h.epoch,h[split+'_loss'],label=split)
        axes[0,1].plot(h.epoch,h[split+'_accuracy'],label=split)
    axes[0,0].set_ylabel('Cross entropy'); axes[0,1].set_ylabel('Accuracy')
    axes[1,0].plot(h.epoch,h.lr); axes[1,0].set_ylabel('Learning rate')
    axes[1,1].plot(h.epoch,h.gradient_norm_mean,label='Mean before clipping')
    axes[1,1].plot(h.epoch,h.gradient_norm_max,label='Max before clipping')
    axes[1,1].axhline(Config.GRAD_CLIP,color='gray',linestyle='--')
    axes[1,1].set_ylabel('Gradient norm')
    for ax in axes.flat:
        ax.axvline(trainer.best_epoch,color='red',linestyle=':',label='Selected epoch')
        ax.set_xlabel('Epoch'); ax.legend(fontsize=8)
    fig.suptitle('Online training uses dropout; checkpoint gap is measured separately in eval mode')
    save_figure(fig,directory/'learning_curves.png')


def model_diagnostics(sid, variant, modality, data, X_train, X_val, normalizer, root):
    directory = root/f'S{sid:02d}'/f'{variant}_{modality}'
    directory.mkdir(parents=True,exist_ok=True)
    seed = Config.MODEL_SEED + 1009*sid
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if Config.DEVICE.type=='cuda':
        torch.cuda.manual_seed_all(seed)
        torch.cuda.reset_peak_memory_stats()
    kwargs = dict(n_channels=X_train.shape[1],n_classes=Config.N_CLASSES,
                  dropout=Config.DROPOUT,variant=variant,modality=modality,n_emg=Config.N_EMG_CH,fs=Config.TARGET_FS)
    model = MultiKernelAttention1DCNN(**kwargs).fit_preprocessor(X_train)
    save_derivative_health(model,X_train,X_val,directory)
    (directory/'architecture.txt').write_text(str(model),encoding='utf-8')
    # Dedicated loader generator gives the same shuffle stream across architectures.
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(EMGWindowDataset(X_train,data['y_train']),
        batch_size=Config.BATCH_SIZE,shuffle=True,generator=generator,num_workers=0)
    val_loader = DataLoader(EMGWindowDataset(X_val,data['y_val']),
        batch_size=Config.BATCH_SIZE,shuffle=False,num_workers=0)
    json_write(directory/'model_config.json',dict(**kwargs,subject=sid,seed=seed,
        selection='minimum validation cross entropy',evaluated_split='validation',
        checkpoint_train_split='four training repetitions',n_params=model.count_params()))
    print(f'\nS{sid:02d} | {variant} | {modality} | {model.count_params():,} parameters | VALIDATION ONLY')
    trainer = DiagnosticTrainer(model,directory/'best_selection.pt',Config.N_CLASSES)
    trainer.fit(train_loader,val_loader)
    learning_report(trainer,directory)
    model.load_state_dict(torch.load(directory/'best_selection.pt',map_location=Config.DEVICE,weights_only=True))
    model.eval()
    tr,_ = prediction_report(model,X_train,data['y_train'],data['meta_train'],directory,'train_eval')
    vl,p = prediction_report(model,X_val,data['y_val'],data['meta_val'],directory,'validation')
    stress = []
    if modality=='both':
        for mask in (('emg','acc','derivative') if variant=='amplitude_velocity' else ('emg','acc')):
            scores,_ = probability_metrics(data['y_val'],predict_arrays(model,X_val,mask=mask))
            stress.append(dict(masked_modality=mask,accuracy=scores['accuracy'],
                               accuracy_drop=vl['accuracy']-scores['accuracy']))
        pd.DataFrame(stress).to_csv(directory/'modality_mask_stress.csv',index=False)
    if Config.DIAG_CHANNEL_MASKS:
        channel_rows=[]
        for ch in range(X_val.shape[1]):
            score,_=probability_metrics(data['y_val'],predict_arrays(model,X_val,mask=ch))
            channel_rows.append(dict(channel=ch,accuracy=score['accuracy'],accuracy_drop=vl['accuracy']-score['accuracy']))
        pd.DataFrame(channel_rows).to_csv(directory/'channel_mask_stress.csv',index=False)
    np.savez_compressed(directory/'normalizer.npz',mean=normalizer.mean,std=normalizer.std)
    summary = dict(subject=sid,variant=variant,modality=modality,
        population='intact' if sid in Config.INTACT_SUBJECTS else 'amputee',
        train_eval_accuracy=tr['accuracy'],validation_accuracy=vl['accuracy'],
        checkpoint_accuracy_gap=tr['accuracy']-vl['accuracy'],validation_f1_macro=vl['f1_macro'],
        validation_nll=vl['nll'],validation_ece=vl['ece_10_bins'],
        high_confidence_error_fraction=vl['high_confidence_error_fraction'],
        selected_epoch=trainer.best_epoch,epochs_run=len(trainer.epoch_rows),
        parameters=model.count_params(),training_seconds=trainer.train_wall,
        peak_cuda_allocated_mb=torch.cuda.max_memory_allocated()/2**20 if Config.DEVICE.type=='cuda' else None)
    json_write(directory/'summary.json',summary)
    flags=[]
    if summary['checkpoint_accuracy_gap']>.10:
        flags.append('Large checkpoint train–validation gap (>10 percentage points): consistent with overfitting and/or repetition distribution shift; not a causal diagnosis.')
    if vl['high_confidence_error_fraction']>.05:
        flags.append('More than 5% of validation windows are wrong with confidence >=90%; inspect confident_errors and calibration.')
    if vl['accuracy']<.75:
        flags.append('Validation accuracy below the heuristic 75% review threshold; inspect the class confusion pairs and signal shifts.')
    if not flags:
        flags.append('No threshold-based flag. This does not prove the model or data are problem-free.')
    (directory/'findings.txt').write_text('\n'.join(flags)+'\nModality/channel masking is an out-of-distribution stress test, not a separately trained ablation.\n',encoding='utf-8')
    print(f"Selected-checkpoint train={tr['accuracy']:.4f}, val={vl['accuracy']:.4f}, gap={summary['checkpoint_accuracy_gap']:.4f}")
    del model,trainer,train_loader,val_loader,p
    gc.collect()
    if Config.DEVICE.type=='cuda': torch.cuda.empty_cache()
    return summary


def write_summary(root, summaries):
    if not summaries:
        return
    frame=pd.DataFrame(summaries)
    frame.to_csv(root/'model_subject_summary.csv',index=False)
    groups=frame.groupby(['variant','modality','population']).agg(
        subjects=('subject','count'),mean_validation_accuracy=('validation_accuracy','mean'),
        std_between_subjects=('validation_accuracy','std'),mean_macro_f1=('validation_f1_macro','mean'),
        mean_checkpoint_gap=('checkpoint_accuracy_gap','mean')).reset_index()
    groups.to_csv(root/'population_summary.csv',index=False)
    paired=frame[frame.modality=='both'].pivot(index='subject',columns='variant',values='validation_accuracy')
    if {'baseline','amplitude_velocity'}.issubset(paired.columns):
        paired=paired.dropna(subset=['baseline','amplitude_velocity'])
        paired['candidate_minus_baseline']=paired.amplitude_velocity-paired.baseline
        paired.to_csv(root/'paired_preprocessing_comparison.csv')
        fig,ax=plt.subplots(figsize=(12,4))
        ax.bar(paired.index.astype(str),100*paired.candidate_minus_baseline)
        ax.axhline(0,color='black'); ax.set(xlabel='Subject',ylabel='Amplitude–velocity minus baseline (percentage points)',title='Paired preprocessing validation comparison; not a test-set result')
        save_figure(fig,root/'paired_preprocessing_comparison.png')
    lines=['# Diagnostic results — validation only','',
        'These measurements identify symptoms; they do not uniquely establish a root cause.', '',
        '## Completed models', '',frame.to_csv(index=False), '',
        '## Read the evidence', '',
        '- The candidate appends 12 standardized EMG derivatives to the original inputs; this is not T-EKIM.',
        '- `derivative_health.csv` checks derivative scale/shift; its normalizer uses training windows only.',
        '- EMG masking removes its derivative information too. Derivative masking leaves original EMG/ACC intact.',
        '- `paired_preprocessing_comparison.csv`: compare models on identical subject/repetition assignments.',
        '- `checkpoint_accuracy_gap`: best-checkpoint train and validation accuracy, both in evaluation mode.',
        '- `signal_health.csv`: large normalized mean/SD changes suggest repetition shift; inspect by class before attributing it to sensor faults.',
        '- `raw_file_audit.json`: file shapes, label ranges, and rerepetition consistency; equal lengths alone do not prove timestamp alignment.',
        '- `validation_confident_errors.csv`: locate wrong windows by source file and sample range.',
        '- `modality_mask_stress.csv`: sensitivity to removing a modality; masking alone cannot establish the value of retraining without that modality.',
        '- `gradient_clipped_fraction` in history: how often clipping occurred; frequent clipping is not automatically a bug.',
        '', 'A validation score was used to select the checkpoint, so it is optimistic as a generalization estimate.',
        'Overlapping windows are correlated; do not treat their count as independent sample size.',
        'The test split was not used for predictions, plots, calibration, or tuning.',
        'Changing normalization, filters, augmentation, or architecture requires a new controlled comparison.',
        '', '## Largest checkpoint gaps','',
        frame.sort_values('checkpoint_accuracy_gap',ascending=False).head(10).to_csv(index=False)]
    (root/'READ_ME_RESULTS.md').write_text('\n'.join(lines),encoding='utf-8')


def diagnostic_preflight(root):
    """Exercise the new trainer and output writers on Kaggle before lengthy fits."""
    directory=root/'_synthetic_preflight'
    directory.mkdir()
    devices=[torch.cuda.current_device()] if Config.DEVICE.type=='cuda' else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(123)
        X=np.random.default_rng(123).normal(size=(4,48,800)).astype(np.float32)
        y=np.array([0,1,0,1],dtype=np.int64)
        metadata=[dict(file_name='SYNTHETIC',source_path='SYNTHETIC',gesture=int(c+Config.GESTURE_MIN),
                       run_index=1,window_start_emg_sample=i*200,window_end_emg_sample=i*200+800)
                  for i,c in enumerate(y)]
        model=MultiKernelAttention1DCNN(48,Config.N_CLASSES,variant='amplitude_velocity',modality='both').fit_preprocessor(X)
        trainer=DiagnosticTrainer(model,directory/'synthetic.pt',Config.N_CLASSES)
        loader=DataLoader(EMGWindowDataset(X,y),batch_size=2,shuffle=False)
        optimizer=Adam(model.parameters(),lr=Config.LR)
        criterion=nn.CrossEntropyLoss()
        trainer._run_epoch(loader,optimizer,criterion)
        trainer._run_epoch(loader,criterion=criterion)
        trainer.best_epoch=1
        learning_report(trainer,directory)
        metrics,_=prediction_report(model,X,y,metadata,directory,'validation')
        normalizer=ChannelNormalizer().fit(X)
        signal_diagnostics(dict(X_train=X,X_val=X),normalizer,directory)
        assert np.isfinite(metrics['nll'])
        assert (directory/'validation_confusion.png').is_file()
        assert (directory/'history.csv').is_file()
        (directory/'SYNTHETIC_ONLY.txt').write_text(
            'Output-generation test passed. These numbers are random synthetic data, not DB7 results.\n',encoding='utf-8')
        del model,trainer,loader,optimizer
    gc.collect()
    if Config.DEVICE.type=='cuda': torch.cuda.empty_cache()
    print('Diagnostic preflight PASS: trainer, gradients, metrics, CSV/NPZ/JSON and PNG writers.')


def run_diagnostics():
    assert not Config.EVALUATE_TEST, 'Diagnostics are restricted to train/validation.'
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    root=Config.KAGGLE_WORKING/f'db7_amplitude_velocity_{stamp}'
    root.mkdir(parents=True,exist_ok=False)
    settings={k:getattr(Config,k) for k in dir(Config) if k.isupper()}
    settings.update(python=sys.version,platform=platform.platform(),torch=torch.__version__,
        numpy=np.__version__,device=str(Config.DEVICE),
        gpu=torch.cuda.get_device_name() if Config.DEVICE.type=='cuda' else None,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        cudnn_benchmark=torch.backends.cudnn.benchmark,
        deterministic_algorithms=torch.are_deterministic_algorithms_enabled(),
        interpretation='validation development; fresh fit per subject; no refit or test evaluation')
    json_write(root/'run_manifest.json',settings)
    summaries=[]
    loader=SubjectLoader()
    global DIAG_RAW_AUDIT
    DIAG_RAW_AUDIT=[]
    print('OUTPUT DIRECTORY:',root)
    print('Training comparisons:',Config.DIAG_EXPERIMENTS)
    try:
        diagnostic_preflight(root)
        for sid in Config.RUN_SUBJECTS:
            subject_root=root/f'S{sid:02d}'
            subject_root.mkdir()
            data,assignment,provenance,n_acc=loader.build_subject_splits(sid)
            json_write(subject_root/'assignments.json',assignment)
            json_write(subject_root/'provenance.json',provenance)
            json_write(root/'raw_file_audit.json',DIAG_RAW_AUDIT)
            integrity_report(data,subject_root)
            for split in ('train','val'):
                # Check every value once, before fitting normalization.
                for start in range(0,len(data['X_'+split]),128):
                    if not np.isfinite(data['X_'+split][start:start+128]).all():
                        raise FloatingPointError(f'S{sid}: nonfinite {split} input before normalization')
                pd.DataFrame(data['meta_'+split]).to_csv(subject_root/f'{split}_window_provenance.csv',index=False)
            counts=[]
            for split in ('train','val','test'):
                labels,ns=np.unique(data['y_'+split],return_counts=True)
                counts.extend(dict(split=split,gesture=int(c+Config.GESTURE_MIN),windows=int(n)) for c,n in zip(labels,ns))
            pd.DataFrame(counts).to_csv(subject_root/'split_class_counts.csv',index=False)
            normalizer=ChannelNormalizer().fit(data['X_train'])
            signal_diagnostics(data,normalizer,subject_root)
            X_train=normalizer.transform(data['X_train'])
            X_val=normalizer.transform(data['X_val'])
            # The diagnostic outputs never need test signal windows or raw arrays again.
            for split in ('train','val','test'):
                del data['X_'+split]
            gc.collect()
            for variant,modality in Config.DIAG_EXPERIMENTS:
                summary=model_diagnostics(sid,variant,modality,data,X_train,X_val,normalizer,root)
                summaries.append(summary)
                write_summary(root,summaries)
            del data,X_train,X_val,normalizer
            gc.collect()
    except Exception:
        (root/'FAILURE.txt').write_text(traceback.format_exc(),encoding='utf-8')
        raise
    finally:
        json_write(root/'raw_file_audit.json',DIAG_RAW_AUDIT)
        write_summary(root,summaries)
        zip_path=root.with_suffix('.zip')
        with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(root.rglob('*')):
                if path.is_file(): archive.write(path,path.relative_to(root))
        print('\nDiagnostic ZIP:',zip_path)
        print('Completed model fits:',len(summaries))
        try:
            from IPython.display import display,FileLink
            display(FileLink(str(zip_path)))
        except ImportError:
            pass
    return root,pd.DataFrame(summaries)


diagnostic_output_dir, diagnostic_summary = run_diagnostics()
